# 01: Feature Engineering, Label Creation, and Smoothing (Price Target)

Refactor objective: pivot from return prediction to **next-day closing price** prediction while preserving exponential smoothing for noise reduction.

## Chapter 1: Introduction & Problem Context

- Domain Overview: We model daily AAPL market behavior using price and technical-indicator features.
- Analytical Objective: Predict a **user-selected next-day target** (configured in this notebook) from today's engineered features.
- Type of ML Problem: Supervised **regression** (continuous target).

In [ ]:
import importlib.metadata  # ensures importlib.metadata is available in some Python envs
import numpy as np
import pandas as pd
import pandas_ta as ta
import yfinance as yf
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 200)


In [ ]:
TICKER = "AAPL"
PERIOD = "5y"
INTERVAL = "1d"
ALPHA = 0.2

OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Shared dataset path consumed by Notebook 02 and Notebook 03
FEATURE_PATH_PARQUET = OUT_DIR / "aapl_features_smoothed.parquet"
FEATURE_PATH_CSV = OUT_DIR / "aapl_features_smoothed.csv"

# Single source-of-truth modeling config for downstream notebooks
MODEL_CONFIG_PATH = OUT_DIR / "modeling_config.json"


## Chapter 2: Dataset & Preprocessing

- Dataset Description: Daily OHLCV from Yahoo Finance via `yfinance`.
- Data Cleaning: Column normalization, date indexing, missing-value handling.
- Feature Engineering: 8 engineered features (mostly unbounded).
- Data Transformation: Exponential smoothing on adjusted close/high/low/volume.
- Train/Test Strategy: Time-aware split will be applied in Notebook 03.

In [ ]:
raw = yf.download(
    TICKER,
    period=PERIOD,
    interval=INTERVAL,
    auto_adjust=False,
    actions=False,
    progress=False,
)

if raw.empty:
    raise ValueError("No data returned from yfinance. Check ticker or network availability.")

if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

df = raw.rename(columns={"Adj Close": "Adj_Close"}).copy()
df = df[["Open", "High", "Low", "Close", "Adj_Close", "Volume"]].dropna().copy()

print("Shape after cleaning:", df.shape)
df.head()

In [ ]:
def exp_smooth(series: pd.Series, alpha: float = 0.2) -> pd.Series:
    return series.ewm(alpha=alpha, adjust=False).mean()

s_adj_close = exp_smooth(df["Adj_Close"], ALPHA)
s_high = exp_smooth(df["High"], ALPHA)
s_low = exp_smooth(df["Low"], ALPHA)
s_volume = exp_smooth(df["Volume"], ALPHA)

In [ ]:
feat = pd.DataFrame(index=df.index)

# Keep core price references
feat["close"] = df["Close"]
feat["adj_close"] = df["Adj_Close"]
feat["adj_close_smooth"] = s_adj_close

# Indicator engineering
feat["rsi_14"] = ta.rsi(s_adj_close, length=14)
macd_df = ta.macd(s_adj_close, fast=12, slow=26, signal=9)
feat["macd_hist_12_26_9"] = macd_df["MACDh_12_26_9"]
feat["macd_line_12_26_9"] = macd_df["MACD_12_26_9"]
feat["macd_signal_12_26_9"] = macd_df["MACDs_12_26_9"]

adx_df = ta.adx(high=s_high, low=s_low, close=s_adj_close, length=14)
feat["adx_14"] = adx_df["ADX_14"]
feat["atr_14"] = ta.atr(high=s_high, low=s_low, close=s_adj_close, length=14)
feat["obv"] = ta.obv(close=s_adj_close, volume=s_volume)
feat["sma_10"] = ta.sma(s_adj_close, length=10)
feat["ema_20"] = ta.ema(s_adj_close, length=20)
feat["cci_20"] = ta.cci(high=s_high, low=s_low, close=s_adj_close, length=20)

bb_df = ta.bbands(s_adj_close, length=20, std=2)
bb_col = next((col for col in bb_df.columns if col.startswith("BBB_20_2")), None)
if bb_col is None:
    raise KeyError(f"Bollinger bandwidth column not found. Available columns: {list(bb_df.columns)}")
feat["bb_bandwidth_20_2"] = bb_df[bb_col]

# Candidate features available for modeling (centralized here)
all_feature_cols = [
    "rsi_14",
    "macd_hist_12_26_9",
    "bb_bandwidth_20_2",
    "adx_14",
    "obv",
    "adj_close_smooth",
    "sma_10",
    "ema_20",
    "macd_line_12_26_9",
    "macd_signal_12_26_9",
    "atr_14",
    "cci_20",
]

# Training feature set (edit ONLY this list to change model inputs)
train_feature_cols = [
    "adj_close_smooth",
    "sma_10",
    "ema_20",
    "macd_line_12_26_9",
    "macd_signal_12_26_9",
    "atr_14",
    "obv",
    "cci_20",
]

# Targets available for training
feat["target_ret_1d_pct"] = (df["Adj_Close"].pct_change().shift(-1) * 100)
feat["target_close_1d"] = df["Close"].shift(-1)
target_options = ["target_ret_1d_pct", "target_close_1d"]

# Training target (edit ONLY this variable to change prediction objective)
target_col = "target_close_1d"

# Safety checks
unknown = sorted(set(train_feature_cols) - set(all_feature_cols))
if unknown:
    raise ValueError(f"train_feature_cols contains unknown features: {unknown}")
if target_col not in target_options:
    raise ValueError(f"target_col must be one of {target_options}, got {target_col}")

keep_cols = ["close", "adj_close"] + all_feature_cols + target_options
feat = feat[keep_cols]

rows_before = len(feat)
feat = feat.dropna().copy()
rows_after = len(feat)

summary = pd.DataFrame({
    "metric": ["rows_before_dropna", "rows_after_dropna", "rows_removed"],
    "value": [rows_before, rows_after, rows_before - rows_after],
})

config_preview = pd.DataFrame({
    "config_item": ["target_col", "n_train_features", "train_features"],
    "value": [target_col, len(train_feature_cols), ", ".join(train_feature_cols)],
})

print("Final modeling shape:", feat.shape)
display(summary)
config_preview


In [ ]:
split_idx = int(len(feat) * 0.8)
train_preview = feat.iloc[:split_idx]
test_preview = feat.iloc[split_idx:]

strategy_tbl = pd.DataFrame({
    "split": ["train", "test"],
    "rows": [len(train_preview), len(test_preview)],
    "start_date": [train_preview.index.min().date(), test_preview.index.min().date()],
    "end_date": [train_preview.index.max().date(), test_preview.index.max().date()],
})
strategy_tbl

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(feat.index, feat["close"], label="Close", alpha=0.75)
ax.plot(feat.index, feat["adj_close_smooth"], label=f"Smoothed Adj Close (alpha={ALPHA})", linewidth=2)
ax.set_title("Close vs Smoothed Adj Close")
ax.set_xlabel("Date")
ax.set_ylabel("Price")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import json

saved_to = []

try:
    feat.to_parquet(FEATURE_PATH_PARQUET)
    saved_to.append(str(FEATURE_PATH_PARQUET))
except Exception:
    feat.to_csv(FEATURE_PATH_CSV, index=True)
    saved_to.append(str(FEATURE_PATH_CSV))

model_config = {
    "data_parquet": str(FEATURE_PATH_PARQUET),
    "data_csv": str(FEATURE_PATH_CSV),
    "all_feature_cols": all_feature_cols,
    "train_feature_cols": train_feature_cols,
    "target_options": target_options,
    "target_col": target_col,
}

MODEL_CONFIG_PATH.write_text(json.dumps(model_config, indent=2), encoding="utf-8")

print("Saved processed feature dataset to:")
for path in saved_to:
    print("-", path)

print("Saved modeling config to:")
print("-", MODEL_CONFIG_PATH)

feat.head()
